# 12 — Chronological inference demo and model card

This notebook shows the smallest credible product view: a ranked incident
queue and a chronological replay of the telemetry and anomaly evidence behind
one incident. It reads only frozen outputs. If an internal diagnostic
configuration was used, every display remains labelled non-deployable.


## 1. Setup and deployment status


In [ ]:
from pathlib import Path
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json

import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

from telco_anomaly.io import (
    file_sha256,
    read_json,
    resolve_data_root,
    write_json,
)

DATA_ROOT = resolve_data_root()
CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v1")
MODEL_RUN_ID = os.getenv("PON_MODEL_RUN_ID", "synthetic_pon_models_v1")
SELECTION_RUN_ID = os.getenv("PON_SELECTION_RUN_ID", "synthetic_pon_selection_v1")
INCIDENT_RUN_ID = os.getenv("PON_INCIDENT_RUN_ID", "synthetic_pon_incidents_v1")
LOCALISATION_RUN_ID = os.getenv("PON_LOCALISATION_RUN_ID", "synthetic_pon_localisation_v1")
EVALUATION_RUN_ID = os.getenv("PON_EVALUATION_RUN_ID", "synthetic_pon_development_v1")

CORE_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID / "SPEC-CORE"
MODEL_ROOT = DATA_ROOT / "models" / "synthetic_pon" / MODEL_RUN_ID
SELECTION_ROOT = DATA_ROOT / "selection" / "synthetic_pon" / SELECTION_RUN_ID
INCIDENT_ROOT = DATA_ROOT / "incidents" / "synthetic_pon" / INCIDENT_RUN_ID
LOCALISATION_ROOT = DATA_ROOT / "localisation" / "synthetic_pon" / LOCALISATION_RUN_ID
EVALUATION_ROOT = DATA_ROOT / "results" / "synthetic_pon" / EVALUATION_RUN_ID

configuration_path = (
    SELECTION_ROOT / "selected_configuration.json"
    if (SELECTION_ROOT / "selected_configuration.json").exists()
    else SELECTION_ROOT / "best_diagnostic_configuration.json"
)
configuration = read_json(configuration_path)
deployment_status = (
    "development gates passed"
    if configuration["status"] == "DEVELOPMENT_GATES_PASSED"
    else "DIAGNOSTIC ONLY — not deployable"
)
incidents = pd.read_parquet(LOCALISATION_ROOT / "localised_incidents.parquet")
members = pd.read_parquet(INCIDENT_ROOT / "incident_members.parquet")
alerts = pd.read_parquet(INCIDENT_ROOT / "alerts.parquet")

display(pd.Series({
    "status": deployment_status,
    "portfolio": configuration["candidate"],
    "incidents": len(incidents),
}, name="product").to_frame())


## 2. Operator incident queue


In [ ]:
queue_columns = [
    "rank", "case_id", "case_start", "case_end", "scope_type", "scope_id",
    "location_ambiguous", "affected_entity_count", "channels",
    "leading_features", "anomaly_evidence_score", "dying_gasp_events",
]
display(incidents[queue_columns].head(30))

if len(incidents):
    chart = px.scatter(
        incidents.head(100),
        x="case_start",
        y="anomaly_evidence_score",
        color="scope_type",
        size="affected_entity_count",
        hover_data=["case_id", "scope_id", "leading_features", "location_ambiguous"],
        title="Ranked Telecom incidents over time",
    )
    chart.show()


## 3. Chronological replay of one incident


In [ ]:
if incidents.empty:
    raise RuntimeError("No incident is available to replay")

CASE_ID = os.getenv("DEMO_CASE_ID", str(incidents.iloc[0]["case_id"]))
selected_case = incidents.loc[incidents["case_id"].astype(str).eq(CASE_ID)].iloc[0]
case_alerts = members.loc[members["case_id"].astype(str).eq(CASE_ID)].merge(
    alerts, on=["alert_id", "entity_id", "model_id"], how="left"
)
entity_id = str(case_alerts.sort_values("peak_score", ascending=False).iloc[0]["entity_id"])
start = pd.to_datetime(selected_case["case_start"], utc=True) - pd.Timedelta(days=1)
end = pd.to_datetime(selected_case["case_end"], utc=True) + pd.Timedelta(days=1)

telemetry_glob = str(CORE_ROOT / "telemetry" / "*.parquet").replace("'", "''")
with duckdb.connect() as connection:
    telemetry = connection.execute(f"""
        SELECT event_ts, metric_id, value
        FROM read_parquet('{telemetry_glob}')
        WHERE CAST(entity_id AS VARCHAR) = ?
          AND event_ts >= ? AND event_ts <= ?
          AND quality_code <> 'invalid'
        ORDER BY event_ts, metric_id
    """, [entity_id, start, end]).df()

metrics_to_show = (
    case_alerts["leading_feature"].dropna().astype(str)
    .str.split("__").str[0].drop_duplicates().head(4).tolist()
)
if not metrics_to_show:
    metrics_to_show = telemetry["metric_id"].drop_duplicates().head(4).tolist()
shown = telemetry.loc[telemetry["metric_id"].isin(metrics_to_show)]

figure = px.line(
    shown,
    x="event_ts", y="value", facet_row="metric_id", color="metric_id",
    title=f"Incident {CASE_ID}: observable telemetry replay for {entity_id}",
)
figure.update_yaxes(matches=None)
figure.add_vrect(
    x0=selected_case["case_start"], x1=selected_case["case_end"],
    fillcolor="red", opacity=0.12, line_width=0,
)
figure.show()
display(case_alerts[[
    "model_id", "alert_start", "alert_end", "peak_score", "leading_feature",
    "evidence_scope_type", "evidence_scope_id",
]])


## 4. Frozen evaluation evidence, when available


In [ ]:
if EVALUATION_ROOT.exists():
    metrics = pd.read_parquet(EVALUATION_ROOT / "metrics.parquet")
    primary_names = {
        "event_recall", "case_precision", "false_cases_per_entity_day",
        "median_detection_delay_seconds", "joint_detection_and_localisation_recall",
    }
    display(metrics.loc[metrics["metric"].isin(primary_names), [
        "metric", "value", "ci_low", "ci_high", "numerator", "denominator",
    ]])
else:
    print("Locked evaluation has not been run; no effectiveness claim is shown.")


## 5. Model card


In [ ]:
model_manifest = read_json(MODEL_ROOT / "model_manifest.json")
model_card = {
    "model_name": "Company-neutral Telecom anomaly and localisation research model",
    "version": "1.0.0",
    "deployment_status": deployment_status,
    "primary_domain": "fixed-access PON",
    "intended_use": [
        "rank anomalous telemetry incidents",
        "consolidate related alerts",
        "estimate the most specific supported physical topology scope",
    ],
    "not_intended_for": [
        "automatic service-impact declaration",
        "fault-type classification",
        "failure prediction",
        "causal root-cause claims",
    ],
    "frozen_channels": configuration["channels"],
    "calibration": model_manifest["threshold_method"],
    "company_independence": (
        "Vendor-neutral semantics and shared algorithms; each operator maps and "
        "calibrates its own data."
    ),
    "evidence_limitations": [
        "Primary labelled results use synthetic PON telemetry.",
        "Public datasets are qualified separately and are never pooled.",
        "Operator data is required before production performance claims.",
        "A diagnostic configuration is not deployable when development gates fail.",
    ],
}
display(pd.Series(model_card, name="model card").to_frame())

MODEL_CARD_PATH = DATA_ROOT / "model_cards" / "synthetic_pon_model_card_v1.json"
if MODEL_CARD_PATH.exists():
    previous = read_json(MODEL_CARD_PATH)
    if previous != model_card:
        raise FileExistsError(f"Refusing to overwrite a different model card: {MODEL_CARD_PATH}")
else:
    write_json(MODEL_CARD_PATH, model_card)
print("Saved model card:", MODEL_CARD_PATH)
